In [94]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

text = """
Language models learn patterns from text.
Transformers use attention to understand context.
Machine learning enables computers to learn patterns.
Deep learning uses neural networks.
Language models predict the next token.
Attention connects information across a sequence.
Artificial intelligence helps solve difficult problems.
Python is useful for building machine learning systems.
A neural network learns by adjusting its parameters.
The next token depends on the context before it.
""" * 150

# Character-level vocabulary
chars = sorted(set(text))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[ch] for ch in s if ch in stoi]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)

split = int(0.9 * len(data))
train_data = data[:split]
val_data = data[split:]

print("Vocabulary size:", vocab_size)
print("Training characters:", len(train_data))
print("Validation characters:", len(val_data))
print("Example IDs:", encode("Language")[:10])


Training device: cpu
Vocabulary size: 34
Training characters: 65745
Validation characters: 7305
Example IDs: [5, 9, 22, 15, 29, 9, 15, 13]


Create input and target sequences

In [95]:

batch_size = 32
block_size = 64  # Maximum context length

def get_batch(split_name):
    source = train_data if split_name == "train" else val_data

    starts = torch.randint(
        len(source) - block_size - 1,
        (batch_size,)
    )

    x = torch.stack([
        source[i:i + block_size] for i in starts
    ])
    y = torch.stack([
        source[i + 1:i + block_size + 1] for i in starts
    ])

    return x.to(device), y.to(device)

xb, yb = get_batch("train")

print("Input batch:", xb.shape)
print("Target batch:", yb.shape)
print("Input example:", decode(xb[0].tolist()))
print("Target example:", decode(yb[0].tolist()))


Input batch: torch.Size([32, 64])
Target batch: torch.Size([32, 64])
Input example: attention to understand context.
Machine learning enables comput
Target example: ttention to understand context.
Machine learning enables compute


Implement causal self-attention

In [96]:

class Head(nn.Module):
    def __init__(self, head_size, d_model, block_size):
        super().__init__()
        self.key = nn.Linear(d_model, head_size, bias=False)
        self.query = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)

        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        scores = q @ k.transpose(-2, -1)
        scores = scores * (k.shape[-1] ** -0.5)

        scores = scores.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        weights = F.softmax(scores, dim=-1)
        return weights @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, d_model, block_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size, d_model, block_size)
            for _ in range(num_heads)
        ])

        self.projection = nn.Linear(
            num_heads * head_size, d_model
        )

    def forward(self, x):
        return self.projection(
            torch.cat([head(x) for head in self.heads], dim=-1)
        )


Build the Transformer block and language model

In [97]:

class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, block_size):
        super().__init__()

        head_size = d_model // num_heads

        self.attention = MultiHeadAttention(
            num_heads, head_size, d_model, block_size
        )

        self.ffn = FeedForward(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=64,
        num_heads=4,
        num_layers=2,
        block_size=64
    ):
        super().__init__()

        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)

        self.blocks = nn.Sequential(*[
            TransformerBlock(d_model, num_heads, block_size)
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        token_emb = self.token_embedding(idx)
        positions = torch.arange(T, device=idx.device)
        position_emb = self.position_embedding(positions)

        x = token_emb + position_emb
        x = self.blocks(x)
        logits = self.lm_head(self.final_norm(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=200):
        self.eval()

        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)

            # Next-token prediction from the final position
            logits = logits[:, -1, :]
            probabilities = F.softmax(logits, dim=-1)

            next_token = torch.multinomial(probabilities, num_samples=1)
            idx = torch.cat((idx, next_token), dim=1)

        return idx


model = TinyGPT(vocab_size).to(device)

print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Model device:", next(model.parameters()).device)

test_x, test_y = get_batch("train")
logits, loss = model(test_x, test_y)

print("Logits shape:", logits.shape)
print("Initial loss:", round(loss.item(), 4))


Parameters: 108194
Model device: cpu
Logits shape: torch.Size([32, 64, 34])
Initial loss: 3.6939


Train the model

In [98]:

learning_rate = 3e-4
max_steps = 500
eval_interval = 100

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=0.01
)

@torch.no_grad()
def estimate_loss():
    model.eval()
    results = {}

    for split_name in ["train", "val"]:
        losses = []

        for _ in range(10):
            x, y = get_batch(split_name)
            _, loss = model(x, y)
            losses.append(loss.item())

        results[split_name] = sum(losses) / len(losses)

    model.train()
    return results


train_history = []
val_history = []

model.train()

for step in range(max_steps):
    xb, yb = get_batch("train")

    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % eval_interval == 0 or step == max_steps - 1:
        metrics = estimate_loss()
        train_history.append(metrics["train"])
        val_history.append(metrics["val"])

        print(
            f"Step {step:4d} | "
            f"Train loss: {metrics['train']:.4f} | "
            f"Validation loss: {metrics['val']:.4f}"
        )


Step    0 | Train loss: 3.6743 | Validation loss: 3.6688
Step  100 | Train loss: 2.5509 | Validation loss: 2.5379
Step  200 | Train loss: 2.1290 | Validation loss: 2.1131
Step  300 | Train loss: 1.8626 | Validation loss: 1.8530
Step  400 | Train loss: 1.5657 | Validation loss: 1.5540
Step  499 | Train loss: 1.2703 | Validation loss: 1.2520


In [102]:
model.eval()

prompt = "Language models"
prompt_ids = encode(prompt)

context = torch.tensor(
    [prompt_ids], dtype=torch.long, device=device
)

generated = model.generate(context, max_new_tokens=200)

print("Generated text:")
print(decode(generated[0].tolist()))


Generated text:
Language modelsd pk texton tion contworrkeatokean.
Trn s.
Deatwork teacrninesxtinsse its isy paneabytsteners.
Lande.
g dis.
A p
Arale lesextttitiners chealeMarnenarke tt fworors arn.
Pyskerkbfobuacue by aritintpal b


Calculate perplexity

In [103]:

import math

final_metrics = estimate_loss()

val_perplexity = math.exp(final_metrics["val"])

print("Validation loss:", round(final_metrics["val"], 4))
print("Validation perplexity:", round(val_perplexity, 4))


Validation loss: 1.2566
Validation perplexity: 3.5133
